In [1]:
import os
import glob
import rasterio
import geopandas as gpd
'''
# 1. Load the shapefile directly (GeoPandas automatically reads the UTM geometry and CRS)
# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids/gce_sites_zone_pixels_centroids.shp")
# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids2/gce_sites_zone_pixels_centroids2.shp")

# 2. Get the native CRS of your biomass models
raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))
with rasterio.open(raster_files[0]) as src:
    raster_crs = src.crs

# 3. Project the points to match the raster if they don't already
if points_gdf.crs != raster_crs:
    points_gdf = points_gdf.to_crs(raster_crs)

# Extract coordinates as a list of (x, y) tuples
coord_tuples = [(pt.x, pt.y) for pt in points_gdf.geometry]

# 4. Loop through the years and sample the data
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Extracting data for year: {year}...")
    
    with rasterio.open(raster_path) as src:
        # Sample the raster and append directly to the GeoDataFrame
        points_gdf[f"biomass_{year}"] = [val[0] for val in src.sample(coord_tuples)]

# 5. Save out the clean results
# You can save it back out as a flat CSV for them, or even a new shapefile/GeoJSON
points_gdf.drop(columns='geometry').to_csv("biomass_extractions_final.csv", index=False)
print("Done!")'''''

'\n# 1. Load the shapefile directly (GeoPandas automatically reads the UTM geometry and CRS)\n# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids/gce_sites_zone_pixels_centroids.shp")\n# points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/gce_sites_zone_pixels_centroids2/gce_sites_zone_pixels_centroids2.shp")\n\n# 2. Get the native CRS of your biomass models\nraster_files = sorted(glob.glob(\'F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif\'))\nwith rasterio.open(raster_files[0]) as src:\n    raster_crs = src.crs\n\n# 3. Project the points to match the raster if they don\'t already\nif points_gdf.crs != raster_crs:\n    points_gdf = points_gdf.to_crs(raster_crs)\n\n# Extract coordinates as a list of (x, y) tuples\ncoord_tuples = [(pt.x, pt.y) for pt in points_gdf.geometry]\n\n# 4. Loop through the years and sample the data\nfor raster_path in raster_fi

In [2]:
points_gdf

NameError: name 'points_gdf' is not defined

In [3]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'rest_trends.csv')

#points_gdf.to_csv(out_csv)

In [13]:
##Window extraction around restoration points
import rasterio
from rasterio.windows import Window
import numpy as np
import pandas as pd
import geopandas as gpd
import glob
import os

# 1. Setup
'''points_gdf = gpd.read_file("C:/Users/arj26323/Documents/Data/Pennings_validation_points/Restoration sites/Rest_sites.shp")
'''
raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))

# Load CSV
# csv_path = "C:/Users/arj26323/Documents/Data/GA_conservation/fixed_actives_installations.csv" #UPDATED LIVINGSHORELINE LOCATIONS

csv_path = "C:/Users/arj26323/Documents/Data/GA_conservation/synthetic_contols/controls.csv" #SYNTHETIC CONTROLS

df = pd.read_csv(csv_path)

# Create GeoDataFrame
points_gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.Longitude, df.Latitude),
    crs="EPSG:4326"
)

'''with rasterio.open(raster_files[0]) as src:
    raster_crs = src.crs
    if points_gdf.crs != raster_crs:
        points_gdf = points_gdf.to_crs(raster_crs)'''

# Define window size (e.g., 2x2)
win_size = 3
pad = win_size // 3

# 2. Loop through files
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Extracting window mean for year: {year}...")
    
    with rasterio.open(raster_path) as src:
        results = []
        for idx, row in points_gdf.iterrows():
            # Get pixel coordinates
            row_idx, col_idx = src.index(row.geometry.x, row.geometry.y)
            
            # Define the window
            window = Window(col_idx - pad, row_idx - pad, win_size, win_size)
            
            # Read the window data
            data = src.read(1, window=window)
            
            # Mask out nodata/NaN values
            # Replace 'src.nodata' with np.nan if your tiff uses actual NaNs
            if src.nodata is not None:
                data = np.where(data == src.nodata, np.nan, data)
                
            # Calculate mean ignoring NaNs
            if np.all(np.isnan(data)):
                results.append(np.nan)
            else:
                results.append(np.nanmean(data))
        
        points_gdf[f"biomass_{year}"] = results

# 3. Save
points_gdf.drop(columns='geometry')
# points_gdf.drop(columns='geometry').to_csv("biomass_extractions_5x5.csv", index=False)
print("Done!")

Extracting window mean for year: 1984...
Extracting window mean for year: 1985...
Extracting window mean for year: 1986...
Extracting window mean for year: 1987...
Extracting window mean for year: 1988...
Extracting window mean for year: 1989...
Extracting window mean for year: 1990...
Extracting window mean for year: 1991...
Extracting window mean for year: 1992...
Extracting window mean for year: 1993...
Extracting window mean for year: 1994...
Extracting window mean for year: 1995...
Extracting window mean for year: 1996...
Extracting window mean for year: 1997...
Extracting window mean for year: 1998...
Extracting window mean for year: 1999...
Extracting window mean for year: 2000...
Extracting window mean for year: 2001...
Extracting window mean for year: 2002...
Extracting window mean for year: 2003...
Extracting window mean for year: 2004...
Extracting window mean for year: 2005...
Extracting window mean for year: 2006...
Extracting window mean for year: 2007...
Extracting windo

In [14]:
points_gdf

,Latitude,Longitude,Site name,year,type,geometry,biomass_1984,biomass_1985,biomass_1986,biomass_1987,...,biomass_2014,biomass_2015,biomass_2016,biomass_2017,biomass_2018,biomass_2019,biomass_2020,biomass_2021,biomass_2022,biomass_2023
0,31.122967,-81.477183,Plantation Creek Site,2010,Oyster Reef,POINT (-81.47718 31.12297),415.569183,596.696716,424.160248,444.760101,...,468.044189,664.676270,467.191315,615.905273,478.254791,434.525177,714.136902,823.788635,479.731964,464.357910
1,31.838650,-81.153333,Florida Passage Site,2012,Oyster Reef,POINT (-81.15333 31.83865),546.787598,628.008118,713.233459,566.339539,...,392.914490,673.458801,477.707764,490.022614,480.411011,418.332520,560.940308,493.276245,459.225952,502.592590
2,32.048017,-81.019233,Oatland Site,2014,Oyster Reef,POINT (-81.01923 32.04802),454.848907,431.684174,551.176636,462.359711,...,409.996338,560.010925,399.928101,434.612732,431.465637,378.262756,505.979706,516.557861,391.161011,401.994263
3,31.988800,-81.024883,Skidaway River Site,2010,Oyster Reef,POINT (-81.02488 31.9888),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,31.326150,-81.322283,Estuaries Area of the Altamaha River,2009,Oyster Reef,POINT (-81.32228 31.32615),432.954224,432.954224,346.108185,367.529572,...,1785.122437,1951.631470,1691.542358,1945.601929,1816.893677,1365.065308,1973.961060,1929.251587,1754.852905,1458.545532
5,31.041650,-81.423017,Jekyll Island Boat Ramp Site,2009,Oyster Reef,POINT (-81.42302 31.04165),426.000031,584.057495,406.831421,387.468353,...,403.886597,589.351074,412.766602,562.577576,411.958008,411.658356,465.735260,575.580261,408.291718,419.075836
6,32.002983,-80.917817,Oyster Creek Site A,2013,Oyster Reef,POINT (-80.91782 32.00298),262.610077,438.078613,411.706024,294.401611,...,270.390839,375.388824,244.519592,265.284393,250.335251,254.687744,255.068298,267.692810,293.230560,261.743652
7,32.004417,-80.914883,Oyster Creek Site B,2013,Oyster Reef,POINT (-80.91488 32.00442),239.060226,372.658813,358.286591,291.562958,...,251.925461,379.952362,205.888474,232.572906,215.731125,246.284225,249.831177,214.575150,207.701630,215.857468
8,32.007950,-80.911683,Oyster Creek Site C,2013,Oyster Reef,POINT (-80.91168 32.00795),325.589691,409.982025,415.601074,346.935638,...,256.266510,361.592194,217.594849,236.406189,253.398880,245.119736,258.031128,235.816574,237.069565,234.777466
9,31.080800,-81.500217,Jointer Creek Oyster Reef A,2012,Oyster Reef,POINT (-81.50022 31.0808),280.419067,399.076416,300.492676,347.091125,...,246.608215,368.636230,251.039337,361.807617,229.932373,265.036041,363.710510,365.909576,254.114487,320.252197


In [5]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'fixed_restoration_window_trends_5.csv')

# points_gdf.to_csv(out_csv)

In [15]:
'''##Visualizing the window at each site and synthetic control

import rasterio
from rasterio.windows import Window

output_dir = "~/Downloads"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
filename = f"window_point_{idx}_year_{year}.gpkg"
output_path = os.path.join(output_dir, filename)

# 1. Start your loops
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    
    with rasterio.open(raster_path) as src:
        # 2. Iterate through each point
        for idx, row in points_gdf.iterrows():
            row_idx, col_idx = src.index(row.geometry.x, row.geometry.y)
            window = Window(col_idx - pad, row_idx - pad, win_size, win_size)
            data = src.read(1, window=window)
            
            # Define the transform for this specific window
            win_transform = src.window_transform(window)
            
            # --- EXPORT LOGIC STARTS HERE ---
            # Create a unique filename for every point-year combo
            filename = f"window_pt{idx}_yr{year}.gpkg"
            output_path = os.path.join(output_dir, filename)
            
            results = (
                {'properties': {'pixel_val': v}, 'geometry': s}
                for s, v in rasterio.features.shapes(data, transform=win_transform)
            )
            
            pixel_gdf = gpd.GeoDataFrame.from_features(list(results), crs=src.crs)
            pixel_gdf.to_file(output_path, driver="GPKG")
'''


In [16]:
# Hotspot data extraction

import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import os
import glob
import pandas as pd

# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_polyg_500k_m2.shp')
# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_savan_100_m2.shp')
# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspot_jekyllish.shp')
# polygons = gpd.read_file('F:/Hotspots/Project shapefiles/hotspots_theilsen_sap_100.shp')
polygons = gpd.read_file('F:/Hotspots/Project shapefiles/all_100.shp')

# polygons = gpd.read_file('F:/GA_marsh_conservation/TLP_boundary.shp') ##NOTE PATHWAY - TLP FILE

raster_files = sorted(glob.glob('F:/Biomass tiffs/Biomass modeling/Georgia/Georgia model only/LOGO_final/*.tif'))

# Initialize an empty list to store results for each polygon
results = []

# 2. Iterate through each raster
for raster_path in raster_files:
    year = os.path.basename(raster_path).split("_")[-1].replace(".tif", "")
    print(f"Processing {year}...")
    
    with rasterio.open(raster_path) as src:
        # Ensure CRS matches
        if polygons.crs != src.crs:
            polygons = polygons.to_crs(src.crs)
            
        # Loop through each polygon
        for idx, row in polygons.iterrows():
            # Use rasterio.mask to clip the raster to the polygon geometry
            # crop=True keeps the resulting array small (the bounding box of the polygon)
            out_image, out_transform = mask(src, [row.geometry], crop=True, nodata=np.nan)
            
            # The mask returns an array [band, height, width]
            data = out_image[0]
            
            # Remove NaNs (NoData pixels)
            valid_pixels = data[~np.isnan(data)]
            
            # Calculate stats
            if valid_pixels.size > 0:
                mean_val = np.nanmean(valid_pixels)
                std_val = np.nanstd(valid_pixels)
            else:
                mean_val = np.nan
                std_val = np.nan
                
            results.append({
                'polygon_id': idx,
                'year': year,
                'mean': mean_val,
                'std': std_val
            })

# 3. Convert results to a DataFrame
stats_df = pd.DataFrame(results)

# Optional: Pivot to wide format if you want one column per year
pivot_df = stats_df.pivot(index='polygon_id', columns='year', values='mean')
print("Extraction complete!")

Processing 1984...
Processing 1985...
Processing 1986...
Processing 1987...
Processing 1988...
Processing 1989...
Processing 1990...
Processing 1991...
Processing 1992...
Processing 1993...
Processing 1994...
Processing 1995...
Processing 1996...
Processing 1997...
Processing 1998...
Processing 1999...
Processing 2000...
Processing 2001...
Processing 2002...
Processing 2003...
Processing 2004...
Processing 2005...
Processing 2006...
Processing 2007...
Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Extraction complete!


In [17]:
stats_df

,polygon_id,year,mean,std
0,0,1984,254.143982,42.131569
1,0,1985,402.975555,67.127029
2,0,1986,268.039551,47.658062
3,0,1987,329.259644,45.774506
4,0,1988,315.414368,59.511745
5,0,1989,389.488739,67.759819
6,0,1990,266.965729,51.324287
7,0,1991,292.584290,74.735641
8,0,1992,378.537872,65.980988
9,0,1993,258.746124,41.870255


In [18]:
##Exporting extraction

out_dir = os.path.expanduser('~/Downloads')
out_csv = os.path.join(out_dir, 'tlp_trend_biomass.csv')

#stats_df.to_csv(out_csv)

In [ ]:
#Fin